In [2]:
import sys
import json
import time
import random
from pathlib import Path
import numpy as np
import pandas as pd
from src.config import ROOT_DIR
from src.pipeline import CampusNLPPipeline

# 1. Define your evaluation dataset
eval_set = [
    {
        "query": "Who is the Director of IIIT Nagpur?",
        "ground_truth": "Prof (Dr) Prem Lal Patel"
    },
    {
        "query": "Who is the HOD of Computer Science and Engineering?",
        "ground_truth": "Dr Milind Penurkar"
    },
    {
        "query": "What is the contact email for the Registrar?",
        "ground_truth": "registrar@iiitn.ac.in"
    }
]

# 2. Save evaluation dataset
eval_path = ROOT_DIR / "data" / "eval_dataset.json"
eval_path.parent.mkdir(parents=True, exist_ok=True)
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_set, f, indent=2)

# 3. Initialize CampusNLP Pipeline
rag = CampusNLPPipeline()
retriever = rag.retriever
print(f"Loaded {len(retriever.chunks)} chunks from IIIT Nagpur corpus.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loaded 1968 chunks from IIIT Nagpur corpus.


In [3]:
import re
from langchain_ollama import ChatOllama

gen_llm = ChatOllama(model="qwen2.5:latest", temperature=0.1, format="json")

candidates = {}
for c in retriever.chunks:
    url = c["source_url"]
    text = c["chunk_text"]
    if url not in candidates and len(text) > 400 and "(cid:" not in text:
        candidates[url] = c

sampled = random.sample(list(candidates.values()), min(22, len(candidates)))

eval_set = []
for idx, doc in enumerate(sampled, 1):
    prompt = f"""Read this IIIT Nagpur document excerpt and write ONE clear question a student or faculty member would ask that can be answered directly from this text.
Return ONLY valid JSON with key "question": {{"question": "..."}}

EXCERPT:
{doc['chunk_text'][:1200]}"""
    try:
        raw = gen_llm.invoke(prompt).content.strip()
        raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
        data = json.loads(raw)
        q = data.get("question") or next(iter(data.values()))
        eval_set.append({
            "question": str(q),
            "expected_url": doc["source_url"],
            "category": doc["category"],
            "answerable": True
        })
    except Exception as e:
        print(f"Row {idx} failed: {e}")

eval_set.extend([
    {"question": "What is today's lunch menu in the IIIT Nagpur hostel mess?", "expected_url": None, "category": "OutOfScope", "answerable": False},
    {"question": "Who won the inter-hostel cricket match yesterday at IIIT Nagpur?", "expected_url": None, "category": "OutOfScope", "answerable": False},
    {"question": "What is the Wi-Fi password for the 4th floor gym at IIIT Nagpur?", "expected_url": None, "category": "OutOfScope", "answerable": False},
])

eval_path = ROOT_DIR / "data" / "eval_dataset.json"
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_set, f, indent=2)

answerable_count = sum(1 for x in eval_set if x["answerable"])
print(f"Saved {len(eval_set)} total questions ({answerable_count} answerable) to {eval_path}")

Saved 25 total questions (22 answerable) to E:\Rasengan\College-Knowledge-Search\data\eval_dataset.json


In [4]:
answerable_queries = [q for q in eval_set if q["answerable"]]

def evaluate_retrieval(method_name, search_fn):
    hit_1, hit_5, mrr_total, latencies = 0, 0, 0.0, []

    for item in answerable_queries:
        t0 = time.perf_counter()
        results = search_fn(item["question"], top_k=5)
        latencies.append((time.perf_counter() - t0) * 1000)

        retrieved_urls = [r["source_url"] for r in results]
        target = item["expected_url"]

        if target in retrieved_urls:
            rank = retrieved_urls.index(target) + 1
            hit_5 += 1
            mrr_total += 1.0 / rank
            if rank == 1:
                hit_1 += 1

    n = len(answerable_queries)
    return {
        "Retrieval Method": method_name,
        "Hit@1 (%)": round((hit_1 / n) * 100, 1),
        "Hit@5 (%)": round((hit_5 / n) * 100, 1),
        "MRR": round(mrr_total / n, 3),
        "Avg Latency (ms)": round(np.mean(latencies), 1)
    }

ablation_df = pd.DataFrame([
    evaluate_retrieval("1. BM25 Only (Sparse)", retriever.search_bm25),
    evaluate_retrieval("2. FAISS Only (Dense)", retriever.search_faiss),
    evaluate_retrieval("3. Hybrid + RRF + Cross-Encoder", retriever.retrieve)
])

display(ablation_df)

,Retrieval Method,Hit@1 (%),Hit@5 (%),MRR,Avg Latency (ms)
0,1. BM25 Only (Sparse),27.3,50.0,0.364,11.0
1,2. FAISS Only (Dense),18.2,50.0,0.308,92.7
2,3. Hybrid + RRF + Cross-Encoder,45.5,50.0,0.466,138.1


In [5]:
import re
from langchain_ollama import ChatOllama

judge_llm = ChatOllama(model="llama3.1:latest", temperature=0, format="json")

eval_logs = []
for item in eval_set:
    res = rag.ask(item["question"])
    context_text = "\n\n".join(s["chunk_text"] for s in res["sources"])

    judge_prompt = f"""Evaluate this RAG response strictly against the provided context.
Return ONLY a valid JSON object with exact lowercase keys:
{{"faithful": true, "refused": false}}

Rules:
- "faithful": true if EVERY claim/number in the answer is directly supported by the context, or if the model properly states the documents lack the info; false if any fact was invented.
- "refused": true if the model stated the documents do not contain enough information.

CONTEXT:
{context_text}

QUESTION: {item['question']}
ANSWER: {res['answer']}"""

    raw = judge_llm.invoke(judge_prompt).content.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    verdict = {k.lower(): v for k, v in json.loads(raw).items()}

    is_faithful = bool(verdict.get("faithful", verdict.get("is_faithful", False)))
    is_refused = bool(verdict.get("refused", verdict.get("is_refused", False)))

    eval_logs.append({
        "Category": item["category"],
        "Question": item["question"],
        "Answerable": item["answerable"],
        "Faithful": is_faithful,
        "Proper Refusal": is_refused == (not item["answerable"]),
        "Answer Preview": res["answer"][:110].replace("\n", " ") + "..."
    })

eval_df = pd.DataFrame(eval_logs)
display(eval_df)

print(f"\nOverall Faithfulness Rate : {eval_df['Faithful'].mean() * 100:.1f}%")
print(f"Refusal Accuracy Rate     : {eval_df['Proper Refusal'].mean() * 100:.1f}%")

,Category,Question,Answerable,Faithful,Proper Refusal,Answer Preview
0,Placements,What languages are listed in the document exce...,True,True,True,The document excerpt does not list any specifi...
1,Admissions,When will the Institute Online Registration fo...,True,True,True,The Institute Online Registration for confirme...
2,General,What is the purpose of publishing an updated G...,True,True,True,The purpose of publishing an updated Guide on ...
3,General,What languages are supported by the Indian Ins...,True,False,True,According to the provided document from the In...
4,Notices,{'question': 'What is the new end date and tim...,True,True,True,The new end date and time for the quotation to...
5,Notices,What languages are listed in the document exce...,True,False,True,The following languages are listed in the docu...
6,Admissions,Will there be a hostel facility available for ...,True,True,True,"Yes, according to the provided context, hostel..."
7,General,What languages are supported by the Indian Ins...,True,False,True,The Indian Institute of Information Technology...
8,General,What languages are listed in the document exce...,True,False,True,The document excerpt lists a wide range of lan...
9,Notices,What criteria must a DPIIT registered Startup ...,True,True,True,A DPIIT registered Startup must meet the quali...



Overall Faithfulness Rate : 60.0%
Refusal Accuracy Rate     : 96.0%


In [6]:
import re
import time
import json
import numpy as np
import pandas as pd
from langchain_ollama import ChatOllama
from src.config import ROOT_DIR

with open(ROOT_DIR / "data" / "eval_dataset.json", "r", encoding="utf-8") as f:
    eval_set = json.load(f)

answerable_queries = [q for q in eval_set if q["answerable"]]

# 1. Retrieval Quality Metrics: Recall@(5, 10, 20), Precision@5, MRR, NDCG@5
r5, r10, r20, p5_list, mrr_list, ndcg5_list = [], [], [], [], [], []
retrieval_latencies, rerank_latencies = [], []

for item in answerable_queries:
    target = item["expected_url"]

    # Measure candidate retrieval (BM25 + FAISS) vs Cross-Encoder re-ranking separately
    t0 = time.perf_counter()
    _ = retriever.search_bm25(item["question"], top_k=20)
    _ = retriever.search_faiss(item["question"], top_k=20)
    t_ret = (time.perf_counter() - t0) * 1000
    retrieval_latencies.append(t_ret)

    t1 = time.perf_counter()
    top20 = retriever.retrieve(item["question"], top_k=20)
    t_tot = (time.perf_counter() - t1) * 1000
    rerank_latencies.append(max(0.0, t_tot - t_ret))

    urls_20 = [r["source_url"] for r in top20]
    urls_10 = urls_20[:10]
    urls_5 = urls_20[:5]

    r5.append(1.0 if target in urls_5 else 0.0)
    r10.append(1.0 if target in urls_10 else 0.0)
    r20.append(1.0 if target in urls_20 else 0.0)

    # Precision@5 (fraction of top-5 chunks matching the relevant source document)
    relevant_in_5 = sum(1 for u in urls_5 if u == target)
    p5_list.append(relevant_in_5 / 5.0)

    # MRR & NDCG@5
    if target in urls_5:
        rank = urls_5.index(target) + 1
        mrr_list.append(1.0 / rank)
        ndcg5_list.append(1.0 / np.log2(rank + 1))
    else:
        mrr_list.append(0.0)
        ndcg5_list.append(0.0)

# 2. Answer Quality & End-to-End Latency
judge_llm = ChatOllama(model="llama3.1:latest", temperature=0, format="json")

faithful_scores, relevance_scores, citation_scores, e2e_latencies = [], [], [], []

for item in eval_set:
    t0 = time.perf_counter()
    res = rag.ask(item["question"])
    e2e_latencies.append((time.perf_counter() - t0) * 1000)

    ans = res["answer"]
    sources = res["sources"]
    context_text = "\n\n".join(f"[Source {i+1}] ({s['source_url']}): {s['chunk_text']}" for i, s in enumerate(sources))

    # Programmatic + LLM check for Citation Accuracy (checks valid [Source N] tags or proper refusal)
    cited_nums = [int(n) for n in re.findall(r"\[Source\s*(\d+)\]", ans)]
    valid_citation_indices = all(1 <= n <= len(sources) for n in cited_nums)

    judge_prompt = f"""Evaluate this RAG answer strictly against the context and question.
Return ONLY valid JSON with exact lowercase keys:
{{"faithful": true, "relevant": true, "citation_accurate": true}}

Rules:
- "faithful": true if every claim in ANSWER is supported by CONTEXT, OR if the context lacks the info and ANSWER properly states it does not have enough information.
- "relevant": true if ANSWER directly addresses the QUESTION (including properly declining when out-of-scope).
- "citation_accurate": true if the cited sources in ANSWER actually support the stated facts (or no false citations are made on a refusal).

CONTEXT:
{context_text}

QUESTION: {item['question']}
ANSWER: {ans}"""

    raw = judge_llm.invoke(judge_prompt).content.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    v = {k.lower(): val for k, val in json.loads(raw).items()}

    faithful_scores.append(bool(v.get("faithful", False)))
    relevance_scores.append(bool(v.get("relevant", False)))
    citation_scores.append(bool(v.get("citation_accurate", False)) and valid_citation_indices)

# 3. Display Slide-Ready Summary Table
slide_summary = pd.DataFrame([
    {"Category": "Retrieval Quality", "Metric": "Recall@5 / Recall@10 / Recall@20", "Value": f"{np.mean(r5)*100:.1f}% / {np.mean(r10)*100:.1f}% / {np.mean(r20)*100:.1f}%"},
    {"Category": "Retrieval Quality", "Metric": "Precision@5", "Value": f"{np.mean(p5_list)*100:.1f}%"},
    {"Category": "Retrieval Quality", "Metric": "MRR", "Value": f"{np.mean(mrr_list):.3f}"},
    {"Category": "Retrieval Quality", "Metric": "NDCG@5", "Value": f"{np.mean(ndcg5_list):.3f}"},
    {"Category": "Answer Quality", "Metric": "Faithfulness / Groundedness", "Value": f"{np.mean(faithful_scores)*100:.1f}%"},
    {"Category": "Answer Quality", "Metric": "Answer Relevance", "Value": f"{np.mean(relevance_scores)*100:.1f}%"},
    {"Category": "Answer Quality", "Metric": "Citation Accuracy", "Value": f"{np.mean(citation_scores)*100:.1f}%"},
    {"Category": "System Performance", "Metric": "Retrieval Latency (BM25 + FAISS)", "Value": f"{np.mean(retrieval_latencies):.1f} ms"},
    {"Category": "System Performance", "Metric": "Re-ranking Latency (Cross-Encoder)", "Value": f"{np.mean(rerank_latencies):.1f} ms"},
    {"Category": "System Performance", "Metric": "End-to-End Latency (Full RAG)", "Value": f"{np.mean(e2e_latencies):.1f} ms"},
])

display(slide_summary)

,Category,Metric,Value
0,Retrieval Quality,Recall@5 / Recall@10 / Recall@20,50.0% / 50.0% / 50.0%
1,Retrieval Quality,Precision@5,14.5%
2,Retrieval Quality,MRR,0.466
3,Retrieval Quality,NDCG@5,0.474
4,Answer Quality,Faithfulness / Groundedness,64.0%
5,Answer Quality,Answer Relevance,64.0%
6,Answer Quality,Citation Accuracy,52.0%
7,System Performance,Retrieval Latency (BM25 + FAISS),32.9 ms
8,System Performance,Re-ranking Latency (Cross-Encoder),92.3 ms
9,System Performance,End-to-End Latency (Full RAG),18346.0 ms


In [16]:
import time

# Place your test query here
query = "Who is the supports Co-ordinator?"

# 1. Retrieve Context & Re-rank (reusing `rag` already loaded in memory)
t0 = time.perf_counter()
sources = rag.retriever.retrieve(query, top_k=5)
retrieval_ms = (time.perf_counter() - t0) * 1000

print(f"=== RETRIEVAL RESULTS ({retrieval_ms:.1f} ms) ===")
for idx, src in enumerate(sources, 1):
    score = src.get("rerank_score") or src.get("score") or src.get("rrf_score")
    score_str = f"{score:.4f}" if isinstance(score, (int, float)) else "N/A"
    print(f"\n[Source {idx}] Score: {score_str} | Category: {src.get('category', 'N/A')}")
    print(f"Title: {src.get('title', 'N/A')}")
    print(f"URL: {src.get('source_url', 'N/A')}")
    print(f"Excerpt: {src.get('chunk_text', '')[:200]}...")

# 2. Generate LLM Answer
print("\n" + "="*50)
print("=== GENERATING ANSWER FROM OLLAMA ===")
t1 = time.perf_counter()
response = rag.ask(query)
total_ms = (time.perf_counter() - t1) * 1000

print(f"\nANSWER ({total_ms:.1f} ms):\n")
print(response["answer"])

=== RETRIEVAL RESULTS (243.5 ms) ===

[Source 1] Score: 0.0156 | Category: Institute
Title: Learn more
URL: https://iiitn.ac.in/media/65bb626447f645ec9264204806ff8911.pdf
Excerpt: Document: Learn more | Category: Institute | Source: https://iiitn.ac.in/media/65bb626447f645ec9264204806ff8911.pdf

Education: Provided that while nominating under clauses (iii), (iv) and (vi), care ...

=== GENERATING ANSWER FROM OLLAMA ===

ANSWER (2706.6 ms):

An officer of the Department of Higher Education, Government of India, not below the rank of Joint Secretary, concerned with technical education, ex officio, acts as the Member-Secretary of the Co-ordination Forum. This individual supports and coordinates the activities of the Co-ordination Forum.

[Source 1]
